# 0.2 — The Oracle Mindset

**Mechanism of the day:** what 'controllable generation' actually *means*, before
we build any fancy model to do it.

Every method later in this gym is an answer to one question: *how do you make a
generator produce samples with a property you want?* To ask that precisely you
need three objects:

- a **generator** `p(x)` — a distribution you can sample sequences from,
- an **oracle** `f(x)` — a function that scores a sequence for some property
  (charge, stability, binding, toxicity...),
- a **target** — the value of `f` you want (e.g. 'high', or '= 5').

Controllable generation = sampling from the **conditional** `p(x | f(x) = y)`
instead of the plain `p(x)`. In this notebook you build all three from scratch
and steer a (toy) generator two ways — **rejection** and **reward tilting** —
and you meet the tension that haunts the entire field: **reward vs diversity**.

Nothing here needs a neural net. That's the point: the *machinery of control* is
separable from the *machinery of generation*. Diffusion, guidance, and RL (the
next tracks) are just better generators plugged into this exact same frame.

**How to use this notebook:** read the concept, implement the reps (they
`raise NotImplementedError`), make the checkpoints (`assert`s) pass. Solutions are
at the very bottom. Tip: in a live/Jupyter session, put `?` after any name (e.g.
`rng.choice?`) to read its docs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (5, 3.2)

AA = 'ACDEFGHIKLMNPQRSTVWY'
N_AA = len(AA)

# Natural amino-acid frequencies (UniProt, %). Our toy generator samples from these.
FREQ = {'A':8.25,'R':5.53,'N':4.06,'D':5.46,'C':1.38,'Q':3.93,'E':6.72,'G':7.07,
        'H':2.27,'I':5.91,'L':9.66,'K':5.80,'M':2.41,'F':3.86,'P':4.74,'S':6.65,
        'T':5.36,'W':1.10,'Y':2.92,'V':6.87}
P = np.array([FREQ[a] for a in AA]); P = P / P.sum()   # probability per amino acid

# Charge at physiological pH (our first oracle uses this).
CHARGE = {'D':-1.0, 'E':-1.0, 'K':1.0, 'R':1.0, 'H':0.1}

# Kyte-Doolittle hydropathy (our second oracle). Positive = hydrophobic.
KD = {'A':1.8,'R':-4.5,'N':-3.5,'D':-3.5,'C':2.5,'Q':-3.5,'E':-3.5,'G':-0.4,
      'H':-3.2,'I':4.5,'L':3.8,'K':-3.9,'M':1.9,'F':2.8,'P':-1.6,'S':-0.8,
      'T':-0.7,'W':-0.9,'Y':-1.3,'V':4.2}
print('setup ready:', N_AA, 'amino acids, freqs sum to', round(P.sum(), 6))

## Part 1 — the generator and the oracle

Our **generator** is deliberately dumb: each residue is drawn independently from
the natural amino-acid frequencies `P`. It knows nothing about the property we
care about. That's realistic — a pretrained protein model also doesn't
*a priori* make what you want; control is what you add on top.

### Rep 1 — `sample_seqs(n, L)`
Draw `n` sequences of length `L`, each residue iid from `P`. Return a list of
strings. (Hint: `rng.choice(N_AA, size=(n, L), p=P)` gives integer indices.)

In [ ]:
def sample_seqs(n, L):
    '''Sample n iid sequences of length L from the background distribution P.'''
    # YOUR CODE HERE
    raise NotImplementedError

# --- checkpoint ---
s = sample_seqs(5, 12)
assert len(s) == 5 and all(len(x) == 12 for x in s)
assert all(c in AA for x in s for c in x)
print('sample:', s[0], '✓')

### Rep 2 — the oracle `net_charge(seq)`
An oracle maps a sequence to a number. Ours: **net charge** = sum of per-residue
charges from `CHARGE` (unlisted residues contribute 0). Real oracles are the
expensive part of the loop — think AlphaFold for 'does it fold', or an assay for
'does it bind'. Here it's one line so we can focus on control.

In [ ]:
def net_charge(seq):
    '''Sum of per-residue charges (CHARGE.get(c, 0.0)).'''
    # YOUR CODE HERE
    raise NotImplementedError

assert net_charge('KKK') == 3.0
assert net_charge('DE') == -2.0
assert net_charge('AAA') == 0.0
print('net_charge works ✓')

## Part 2 — what does the generator produce on its own?

Before steering, look at the **baseline**: sample a big pool and histogram the
property. This is `p(f(x))` under the unconditional generator — the thing we want
to shift.

In [ ]:
L = 30
pool = sample_seqs(20000, L)
charges = np.array([net_charge(s) for s in pool])
print('baseline net charge: mean %.2f, std %.2f' % (charges.mean(), charges.std()))

plt.hist(charges, bins=range(-12, 13), color='#888', edgecolor='white')
plt.axvline(charges.mean(), color='k', ls='--', label='mean')
plt.title('baseline p(net charge)'); plt.xlabel('net charge'); plt.legend(); plt.show()

The baseline sits near 0 (D/E cancel K/R). **Goal:** we want strongly *positive*
peptides — say net charge >= 6 — which are rare under `p(x)`. Two ways to get
them, in increasing sophistication.

## Method 1 — rejection sampling (filtering)

The simplest possible control: **sample from `p(x)`, throw away anything that
fails** `f(x) >= threshold`. The survivors are exact samples from the conditional
`p(x | f(x) >= threshold)`. Correct, and dead simple.

The catch is *cost*: if only a fraction `a` of samples pass, you need ~`1/a` draws
per accepted sequence, and `a` shrinks fast as the target gets more extreme. This
is exactly why we need smarter methods — and why 'best-of-N' has limits.

### Rep 3 — `rejection_filter(seqs, oracle, threshold)`
Return `(kept, accept_rate)`: the subset with `oracle(s) >= threshold`, and the
fraction accepted.

In [ ]:
def rejection_filter(seqs, oracle, threshold):
    '''Return (kept_seqs, accept_rate) for oracle(s) >= threshold.'''
    # YOUR CODE HERE
    raise NotImplementedError

for thr in [2, 4, 6, 8]:
    kept, rate = rejection_filter(pool, net_charge, thr)
    assert all(net_charge(s) >= thr for s in kept)
    assert 0.0 <= rate <= 1.0
    cost = float('inf') if rate == 0 else round(1 / rate)
    print('charge >= %d : accept %.3f  (~%s draws per hit)' % (thr, rate, cost))
print('notice the cost exploding as the target gets extreme ✓')

## Method 2 — reward tilting (the idea behind everything)

Instead of hard accept/reject, **softly reweight** the whole distribution toward
high reward. Define a reward `r(x)` (here `r = net_charge`) and target the
**tilted distribution**

```
p_T(x)  ~  p(x) * exp( r(x) / T )
```

`T` is a temperature. `T -> infinity` gives back the base `p(x)` (no steering);
`T -> 0` concentrates all mass on the single best `x` (pure greedy / best-of-N).
In between you trade off. This one equation is the skeleton of **classifier
guidance**, **RLHF / DPO**, and **energy-based steering** — they all approximate
sampling from `p(x) * exp(reward/T)` with a real generator.

We approximate it here by **importance resampling**: draw a big pool from `p(x)`,
then resample it with weights `w(x) = softmax(r(x) / T)`.

### Rep 4 — `tilted_resample(seqs, rewards, T, k)`
Compute weights `w = softmax(rewards / T)` and draw `k` indices with
`rng.choice(len(seqs), size=k, p=w)`. Return `(resampled_seqs, idx)`.
(Use the max-subtraction trick for a numerically stable softmax.)

In [ ]:
def tilted_resample(seqs, rewards, T, k):
    '''Importance-resample seqs with weights softmax(rewards / T). Returns (seqs_out, idx).'''
    # YOUR CODE HERE
    raise NotImplementedError

out, idx = tilted_resample(pool, charges, T=0.7, k=5000)
assert len(out) == 5000
shifted = charges[idx].mean()
assert shifted > charges.mean() + 3, 'tilting should raise mean charge a lot'
print('baseline mean %.2f -> tilted(T=0.7) mean %.2f  ✓' % (charges.mean(), shifted))

## Part 3 — the tension: reward vs diversity

Cranking `T` down keeps raising the average reward — but the samples collapse
onto a few high-reward sequences. **Reward up, diversity down.** This tradeoff is
*the* recurring headache of controllable generation: over-optimize an oracle and
you get degenerate, non-diverse, often oracle-hacking outputs. Let's measure it.

### Rep 5 — `diversity(seqs)`
Return the fraction of *unique* sequences: `#unique / #total` (1.0 = all
distinct, near 0 = collapsed onto repeats).

In [ ]:
def diversity(seqs):
    '''Fraction of unique sequences.'''
    # YOUR CODE HERE
    raise NotImplementedError

assert diversity(['A','A','B']) == 2/3
assert diversity(['A','B','C']) == 1.0
print('diversity works ✓')

In [ ]:
# Sweep temperature and plot the reward-diversity frontier.
Ts = [1000, 8, 4, 2, 1.2, 0.8, 0.5, 0.3]
mean_reward, div = [], []
for T in Ts:
    out, idx = tilted_resample(pool, charges, T=T, k=5000)
    mean_reward.append(charges[idx].mean())
    div.append(diversity(out))

fig, ax = plt.subplots()
ax.plot(mean_reward, div, '-o')
for T, mr, d in zip(Ts, mean_reward, div):
    ax.annotate('T=%g' % T, (mr, d), fontsize=8, xytext=(4, 2), textcoords='offset points')
ax.set_xlabel('mean reward (net charge)'); ax.set_ylabel('diversity (unique frac)')
ax.set_title('the reward vs diversity frontier'); plt.show()
print('every controllable-generation method lives somewhere on a curve like this ✓')

### The Bayesian view (why `exp(r/T)` is not arbitrary)

If the property is a label `y` with likelihood `p(y | x)`, then Bayes says
`p(x | y) ~ p(x) * p(y | x)`. Writing `p(y | x) = exp(r(x)/T)` recovers exactly
our tilted distribution — the reward is a log-likelihood, and `T` sets how
strictly we insist on the property. So:

- **Rejection** = the hard-constraint special case (`p(y|x)` is 0/1).
- **Best-of-N** = tilting in the `T -> 0` limit, approximated with N draws.
- **Classifier guidance** (Track 2) = pushing a diffusion model's samples up the
  gradient of `log p(y | x)` — the differentiable version of this reweighting.
- **RLHF / DPO** (Track 2) = *training* a generator so its own `p(x)` becomes the
  tilted `p(x) exp(r/T)`, instead of resampling after the fact.

Same target distribution, different ways to reach it. You just built the target.

## Part 4 — protein instance: a multi-property design

Real design wants several properties at once. Let's ask for a peptide that is
**both positively charged and hydrophobic** (a cartoon of a membrane-active /
antimicrobial peptide). Combine two oracles into one reward and steer with it —
exactly how multi-objective design weights competing scores in practice.

### Rep 6 — `mean_hydropathy(seq)`
Return the mean Kyte-Doolittle value over the sequence (`KD[c]` per residue).

In [ ]:
def mean_hydropathy(seq):
    '''Mean Kyte-Doolittle hydropathy over the sequence.'''
    # YOUR CODE HERE
    raise NotImplementedError

assert abs(mean_hydropathy('IIII') - 4.5) < 1e-9
assert mean_hydropathy('RRRR') < 0
print('mean_hydropathy works ✓')

In [ ]:
# Combined reward: reward charge AND hydrophobicity (weights balance their scales).
def reward(seq):
    return 1.0 * net_charge(seq) + 2.0 * mean_hydropathy(seq)

rewards = np.array([reward(s) for s in pool])
out, idx = tilted_resample(pool, rewards, T=1.0, k=5000)

base_c, base_h = charges.mean(), np.array([mean_hydropathy(s) for s in pool]).mean()
des_c = np.array([net_charge(s) for s in out]).mean()
des_h = np.array([mean_hydropathy(s) for s in out]).mean()
print('property         baseline -> designed')
print('net charge        %6.2f -> %6.2f' % (base_c, des_c))
print('mean hydropathy   %6.2f -> %6.2f' % (base_h, des_h))
print('example designed peptide:', max(out, key=reward))
print('both properties moved up together, from ONE reward signal ✓')

## Reflection — what just transferred

- **The frame `(p, f, target)`** is the whole game. Every later notebook swaps in
  a better `p` (autoregressive, diffusion, flow) or a better way to hit `f`.
- **`p(x) exp(r/T)`** is the universal target of steering: guidance, best-of-N,
  RLHF and DPO are all routes to this same tilted distribution.
- **Reward vs diversity** is the tradeoff you will fight forever; watching for
  mode collapse / oracle-hacking is a core skill.
- **Oracles are the bottleneck.** Ours was free; in reality `f` is a slow assay or
  a big model (ESM, AlphaFold), which is why cheap surrogate oracles and
  active learning matter (Track 3).

**Next rep:** `1.1 Autoregressive sequence model from scratch` — replace the dumb
iid generator with a real learned `p(x)` you can sample, so 'better generator'
stops being a toy.

---
Scroll down only after you've done the reps.

## Solutions appendix (peek only after trying)

In [ ]:
def sample_seqs(n, L):
    idx = rng.choice(N_AA, size=(n, L), p=P)
    return [''.join(AA[i] for i in row) for row in idx]

def net_charge(seq):
    return sum(CHARGE.get(c, 0.0) for c in seq)

def rejection_filter(seqs, oracle, threshold):
    scores = np.array([oracle(s) for s in seqs])
    keep = scores >= threshold
    return [s for s, k in zip(seqs, keep) if k], float(keep.mean())

def tilted_resample(seqs, rewards, T, k):
    w = np.asarray(rewards, dtype=float) / T
    w = np.exp(w - w.max())
    w = w / w.sum()
    idx = rng.choice(len(seqs), size=k, p=w)
    return [seqs[i] for i in idx], idx

def diversity(seqs):
    return len(set(seqs)) / len(seqs)

def mean_hydropathy(seq):
    return float(np.mean([KD[c] for c in seq]))

print('reference solutions loaded — re-run the checkpoint cells above')